<div style="background-color: #FFDDDD; border-left: 5px solid red; padding: 10px; color: black;">
    <strong>Kernel:</strong> Python 3 (ipykernel)
</div>

# 🚀 Deploy `Qwen/Qwen3-4B-Instruct-2507` on Amazon SageMaker

## Prerequisites

To start off, let's install some packages to help us through the notebooks. **Restart the kernel after packages have been installed.**

In [1]:
%pip install -r ./scripts/requirements.txt --upgrade

INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/

## This cell will restart the kernel. Click "OK".

In [2]:
from IPython import get_ipython
get_ipython().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

***

In [1]:
import os
import json
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.config import load_sagemaker_config
from sagemaker.core.common_utils import name_from_base

sagemaker_session = Session()
s3_client = boto3.client('s3')

region = sagemaker_session.boto_session.region_name
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix
configs = load_sagemaker_config()

role = get_execution_role(sagemaker_session, use_default=True)

print(f"Execution Role: {role}")
print(f"Default S3 Bucket: {bucket_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Execution Role: arn:aws:iam::590184118038:role/workshop-executionrole
Default S3 Bucket: sagemaker-us-east-1-590184118038


## Deploy Model to SageMaker Hosting

### Step 1: Get SageMaker LMI Container to host Qwen

In [2]:
inference_image_uri = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:0.33.0-lmi15.0.0-cu128"
print(f"Using image to host: {inference_image_uri}")

Using image to host: 763104351884.dkr.ecr.us-east-1.amazonaws.com/djl-inference:0.33.0-lmi15.0.0-cu128


### Step 2: Deploy model using v3 SageMaker SDK

In [3]:
model_id = "Qwen/Qwen3-4B-Instruct-2507"
model_id_filesafe = model_id.replace("/","_")

use_local_model = True #set to false for the training job to download from HF, otherwise True will download locally

In [4]:
from huggingface_hub import snapshot_download
import os
import subprocess

if use_local_model:
    model_local_location = f"../models/{model_id_filesafe}"
    prefix = f"{default_prefix}/" if default_prefix else ""
    model_s3_destination = f"s3://{bucket_name}/{prefix}models/{model_id_filesafe}"
    
    print(f"Downloading model {model_id}")
    os.makedirs(model_local_location, exist_ok=True)
    snapshot_download(repo_id=model_id, local_dir=model_local_location)
    
    print("Beginning Model Upload...")
    subprocess.run(['aws', 's3', 'sync', model_local_location, model_s3_destination, 
                   '--exclude', '.cache/*', '--exclude', '.gitattributes'])
    
    print(f"Model uploaded to:\n{model_s3_destination}")
    os.environ["model_location"] = model_s3_destination
else:
    os.environ["model_location"] = model_id

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Beginning Model Upload...
upload: ../models/Qwen_Qwen3-4B-Instruct-2507/LICENSE to s3://sagemaker-us-east-1-590184118038/models/Qwen_Qwen3-4B-Instruct-2507/LICENSE
upload: ../models/Qwen_Qwen3-4B-Instruct-2507/README.md to s3://sagemaker-us-east-1-590184118038/models/Qwen_Qwen3-4B-Instruct-2507/README.md
upload: ../models/Qwen_Qwen3-4B-Instruct-2507/config.json to s3://sagemaker-us-east-1-590184118038/models/Qwen_Qwen3-4B-Instruct-2507/config.json
upload: ../models/Qwen_Qwen3-4B-Instruct-2507/generation_config.json to s3://sagemaker-us-east-1-590184118038/models/Qwen_Qwen3-4B-Instruct-2507/generation_config.json
upload: ../models/Qwen_Qwen3-4B-Instruct-2507/model.safetensors.index.json to s3://sagemaker-us-east-1-590184118038/models/Qwen_Qwen3-4B-Instruct-2507/model.safetensors.index.json
upload: ../models/Qwen_Qwen3-4B-Instruct-2507/tokenizer_config.json to s3://sagemaker-us-east-1-590184118038/models/Qwen_Qwen3-4B-Instruct-2507/tokenizer_config.json
upload: ../models/Qwen_Qwen3-4B-In

In [5]:
inference_llm_config = {
    "HF_MODEL_ID": os.environ["model_location"],
    "OPTION_MAX_MODEL_LEN": "4096",
    "OPTION_GPU_MEMORY_UTILIZATION": "0.8",
    "OPTION_ENABLE_STREAMING": "false",
    "OPTION_ROLLING_BATCH": "vllm",
    "OPTION_MODEL_LOADING_TIMEOUT": "3600",
    "OPTION_PAGED_ATTENTION": "false",
    'OPTION_TRUST_REMOTE_CODE': 'true',
    'OPTION_DTYPE': 'bf16',
    'OPTION_QUANTIZE': 'fp8',
    'OPTION_TENSOR_PARALLEL_DEGREE': 'max',
    'OPTION_MAX_ROLLING_BATCH_SIZE': '32',
}

In [6]:
from sagemaker.core.resources import Model, Endpoint, EndpointConfig
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

model_name = "Qwen3-4B-Instruct-2507"

core_model = Model.create(
    model_name=model_name,
    execution_role_arn=role,
    primary_container=ContainerDefinition(
        image=inference_image_uri,
        environment=inference_llm_config,
    ),
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[06/05/26 04:07:25] INFO     Creating model resource.                                            ]8;id=702622;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=625569;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#20477\20477]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=847934;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=545050;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#361\361]8;;\

                    INFO     Runs on sagemaker prod, region:us-east-1                                  ]8;id=64876;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=644970;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#375\375]8;;\

In [7]:
from sagemaker.core.helper.session_helper import _wait_until, _deploy_done

endpoint_name = f"{model_name}-endpoint"
BASE_ENDPOINT_NAME = name_from_base(endpoint_name)

EndpointConfig.create(
    endpoint_config_name=BASE_ENDPOINT_NAME,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=model_name,
            initial_instance_count=1,
            instance_type="ml.g5.2xlarge",
        )
    ],
)

core_endpoint = Endpoint.create(
    endpoint_name=BASE_ENDPOINT_NAME,
    endpoint_config_name=BASE_ENDPOINT_NAME,
)

_wait_until(lambda: _deploy_done(sagemaker_session.sagemaker_client, BASE_ENDPOINT_NAME), poll=30)
core_endpoint = Endpoint.get(endpoint_name=BASE_ENDPOINT_NAME)
print(f"Endpoint status: {core_endpoint.endpoint_status}")

[06/05/26 04:07:28] INFO     Creating endpoint_config resource.                                  ]8;id=601296;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=24943;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#11069\11069]8;;\

                    INFO     Creating endpoint resource.                                         ]8;id=406114;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=978731;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#10228\10228]8;;\

-------------------!Endpoint status: InService


In [12]:
SYSTEM_PROMPT = """You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week."""

USER_PROMPT = """[Instruction]
Analyze the weekly stock price movement for Apple (AAPL) based on the following recent news and financials.
[Input]
Recent news: Apple reported quarterly earnings that beat analyst estimates, driven by strong iPhone 15 sales and services revenue growth. The company announced an expanded share buyback program worth $90 billion.
Basic financials: Gross margin 45.2%, operating margin 30.1%, P/E ratio 28.4, revenue growth YoY +6%.
[Predict the weekly price change for Apple.]
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

messages

[{'role': 'system',
  'content': "You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week."},
 {'role': 'user',
  'content': '[Instruction]\nAnalyze the weekly stock price movement for Apple (AAPL) based on the following recent news and financials.\n[Input]\nRecent news: Apple reported quarterly earnings that beat analyst estimates, driven by strong iPhone 15 sales and services revenue growth. The company announced an expanded share buyback program worth $90 billion.\nBasic financials: Gross margin 45.2%, operating margin 30.1%, P/E ratio 28.4, revenue growth YoY +6%.\n[Predict the weekly price change for Apple.]\n'}]

In [13]:
payload = json.dumps({
    "messages": messages,
    "parameters": {
        "temperature": 0.2,
        "top_p": 0.9,
        "return_full_text": False,
        "max_new_tokens": 1024,
    },
})

response = core_endpoint.invoke(
    body=payload,
    content_type="application/json",
    accept="application/json",
)

result = json.loads(response.body.read().decode("utf-8"))
result["choices"][0]["message"]["content"]

"**Weekly Stock Price Movement Prediction for Apple (AAPL)**\n\n**Analysis:**\n\nApple (AAPL) reported strong quarterly earnings that exceeded analyst expectations, primarily driven by:\n\n- **Strong iPhone 15 sales**: Indicates continued demand for Apple's flagship product, signaling robust consumer confidence and effective product positioning.\n- **Services revenue growth**: A key positive trend—services (like App Store, Apple Music, iCloud) are growing at a faster pace than hardware, reflecting improved customer retention and ecosystem stickiness.\n- **Expanded share buyback program ($90 billion)**: This signals confidence in the company’s long-term value and strengthens investor sentiment by increasing shareholder returns through reduced share count.\n\n**Financial Highlights:**\n- **Gross margin of 45.2%**: Suggests efficient cost management and pricing power, especially in a competitive hardware market.\n- **Operating margin of 30.1%**: Indicates strong profitability and operatio

In [16]:
from IPython.display import Markdown
Markdown(result["choices"][0]["message"]["content"])

**Weekly Stock Price Movement Prediction for Apple (AAPL)**

**Analysis:**

Apple (AAPL) reported strong quarterly earnings that exceeded analyst expectations, primarily driven by:

- **Strong iPhone 15 sales**: Indicates continued demand for Apple's flagship product, signaling robust consumer confidence and effective product positioning.
- **Services revenue growth**: A key positive trend—services (like App Store, Apple Music, iCloud) are growing at a faster pace than hardware, reflecting improved customer retention and ecosystem stickiness.
- **Expanded share buyback program ($90 billion)**: This signals confidence in the company’s long-term value and strengthens investor sentiment by increasing shareholder returns through reduced share count.

**Financial Highlights:**
- **Gross margin of 45.2%**: Suggests efficient cost management and pricing power, especially in a competitive hardware market.
- **Operating margin of 30.1%**: Indicates strong profitability and operational efficiency.
- **P/E ratio of 28.4**: While elevated, it reflects market expectations of sustained growth and profitability. The current valuation is justified given the strong fundamentals and buyback commitment.
- **Revenue growth of +6% YoY**: Solid performance, especially in a macroeconomic environment with inflation and consumer spending caution.

**Market Sentiment:**
The combination of beat earnings, robust margins, and a major buyback initiative has likely triggered positive investor sentiment. Institutional investors and retail traders are likely to view this as a sign of long-term strength and strategic confidence.

**Predicted Weekly Price Change:**

👉 **Apple (AAPL) is expected to see a positive weekly price movement of +1.5% to +2.5%**.

This range reflects:
- A moderate bullish reaction to the earnings beat and buyback announcement.
- Market pricing in the higher P/E, which may lead to some volatility but generally supports upside if fundamentals remain strong.
- Continued demand for Apple’s ecosystem and services growth as a tailwind.

**Conclusion:**
With strong fundamentals, a clear strategic move (buyback), and a solid earnings performance, Apple is well-positioned for a **positive weekly stock price movement**. Investors should remain cautious about macroeconomic headwinds, but in the near term, the news supports a bullish outlook.

**Prediction: +2.0% weekly price increase (target range: +1.5% to +2.5%)**.

### Store variables

Save the endpoint name for use later

In [14]:
%store BASE_ENDPOINT_NAME

Stored 'BASE_ENDPOINT_NAME' (str)
